# Day 081 — Exercise 1: The Toolbox and Argument Validation

**What you'll build:** a toolbox where every tool declares a *typed parameter schema*, plus `validate_args` to check arguments before a tool runs.

**Why it matters:** with many tools, the model *will* sometimes pick the right tool but pass the wrong arguments — a missing field, or a word where a number belongs. Validating against the schema catches that before the tool throws, and gives the agent a clear error to recover from.

In [ ]:
import json

def _mock_pick(tool, args):
    """Return an llm_fn that always routes to `tool` with `args` (as JSON)."""
    payload = json.dumps({'tool': tool, 'args': args})
    return lambda messages: payload
import ast
import json
import operator

# ── helpers reused from Days 79-80 ───────────────────────────────────────────
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("unsupported expression")


def safe_calculate(expression):
    """Evaluate arithmetic without eval() (Day 79)."""
    return _eval_node(ast.parse(expression, mode="eval").body)


def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]


## Task

1. `DEFAULT_TOOLS` — a list of tool dicts, each `{'name', 'description', 'parameters', 'fn'}`. Every parameter is `{'type', 'required', 'description'}`. Include `calculator`, `word_count`, `uppercase`, `reverse`, and `repeat` (`repeat` takes `text: string` and `times: integer`).
2. `validate_args(tool, args) -> (ok, error)` — for each declared parameter: if it's `required` and missing → fail; if present, check its value against the declared `type` (`string`, `integer`, `number`, `boolean`). Return `(True, '')` when everything checks out.

## Your Implementation

In [ ]:
def validate_args(tool, args):
    """Validate args against tool['parameters']. Returns (ok: bool, error: str)."""
    raise NotImplementedError

DEFAULT_TOOLS = []  # calculator, word_count, uppercase, reverse, repeat


In [ ]:

# ── typed parameter schemas + argument validation ────────────────────────────
def _is_number(s):
    try:
        float(s)
        return True
    except (TypeError, ValueError):
        return False


# each check answers: does this value satisfy the declared type?
_TYPE_CHECKS = {
    "string": lambda v: isinstance(v, str),
    "integer": lambda v: (isinstance(v, int) and not isinstance(v, bool))
                         or (isinstance(v, str) and v.strip().lstrip("-").isdigit()),
    "number": lambda v: (isinstance(v, (int, float)) and not isinstance(v, bool))
                        or (isinstance(v, str) and _is_number(v)),
    "boolean": lambda v: isinstance(v, bool),
}


def validate_args(tool, args):
    """Validate args against a tool's parameter schema.

    Returns (ok: bool, error: str). Checks two things per declared parameter:
    required parameters must be present, and present values must match the
    declared type. An unknown declared type is treated as no constraint.
    """
    for pname, pspec in tool.get("parameters", {}).items():
        if pspec.get("required") and pname not in args:
            return False, "missing required parameter: " + pname
        if pname in args:
            check = _TYPE_CHECKS.get(pspec.get("type"))
            if check is not None and not check(args[pname]):
                return False, "parameter " + pname + " must be " + str(pspec.get("type"))
    return True, ""


# ── the toolbox: each tool declares name, description, typed schema, fn ───────
def _p(type_, required=True, description=""):
    """Shorthand for a parameter spec."""
    return {"type": type_, "required": required, "description": description}


DEFAULT_TOOLS = [
    {"name": "calculator",
     "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",
     "parameters": {"expression": _p("string", description="the arithmetic")},
     "fn": lambda args: str(safe_calculate(args["expression"]))},
    {"name": "word_count",
     "description": "Count the words in a piece of text.",
     "parameters": {"text": _p("string", description="text to count")},
     "fn": lambda args: str(len(str(args["text"]).split()))},
    {"name": "uppercase",
     "description": "Convert text to UPPERCASE.",
     "parameters": {"text": _p("string", description="text to upcase")},
     "fn": lambda args: str(args["text"]).upper()},
    {"name": "reverse",
     "description": "Reverse a piece of text.",
     "parameters": {"text": _p("string", description="text to reverse")},
     "fn": lambda args: str(args["text"])[::-1]},
    {"name": "repeat",
     "description": "Repeat a piece of text N times.",
     "parameters": {"text": _p("string", description="text to repeat"),
                    "times": _p("integer", description="how many times")},
     "fn": lambda args: str(args["text"]) * int(args["times"])},
]


## Automated checks

In [ ]:

score, total = 0, 5
try:
    names = {t['name'] for t in DEFAULT_TOOLS}
    assert {'calculator', 'word_count', 'uppercase', 'reverse', 'repeat'} <= names
    score += 1; print("✅ the toolbox has the expected tools")

    repeat = next(t for t in DEFAULT_TOOLS if t['name'] == 'repeat')
    assert repeat['fn']({'text': 'ab', 'times': 3}) == 'ababab'
    score += 1; print("✅ tool functions run")

    assert repeat['parameters']['times']['required'] is True
    assert repeat['parameters']['times']['type'] == 'integer'
    score += 1; print("✅ parameters carry a schema (type + required)")

    ok, err = validate_args(repeat, {'text': 'hi', 'times': 2})
    assert ok and err == ''
    score += 1; print("✅ validate_args accepts valid args")

    miss, _ = validate_args(repeat, {'text': 'hi'})
    wrong, _ = validate_args(repeat, {'text': 'hi', 'times': 'lots'})
    assert miss is False and wrong is False
    score += 1; print("✅ validate_args rejects missing and wrong-type args")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── typed parameter schemas + argument validation ────────────────────────────
def _is_number(s):
    try:
        float(s)
        return True
    except (TypeError, ValueError):
        return False


# each check answers: does this value satisfy the declared type?
_TYPE_CHECKS = {
    "string": lambda v: isinstance(v, str),
    "integer": lambda v: (isinstance(v, int) and not isinstance(v, bool))
                         or (isinstance(v, str) and v.strip().lstrip("-").isdigit()),
    "number": lambda v: (isinstance(v, (int, float)) and not isinstance(v, bool))
                        or (isinstance(v, str) and _is_number(v)),
    "boolean": lambda v: isinstance(v, bool),
}


def validate_args(tool, args):
    """Validate args against a tool's parameter schema.

    Returns (ok: bool, error: str). Checks two things per declared parameter:
    required parameters must be present, and present values must match the
    declared type. An unknown declared type is treated as no constraint.
    """
    for pname, pspec in tool.get("parameters", {}).items():
        if pspec.get("required") and pname not in args:
            return False, "missing required parameter: " + pname
        if pname in args:
            check = _TYPE_CHECKS.get(pspec.get("type"))
            if check is not None and not check(args[pname]):
                return False, "parameter " + pname + " must be " + str(pspec.get("type"))
    return True, ""


# ── the toolbox: each tool declares name, description, typed schema, fn ───────
def _p(type_, required=True, description=""):
    """Shorthand for a parameter spec."""
    return {"type": type_, "required": required, "description": description}


DEFAULT_TOOLS = [
    {"name": "calculator",
     "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",
     "parameters": {"expression": _p("string", description="the arithmetic")},
     "fn": lambda args: str(safe_calculate(args["expression"]))},
    {"name": "word_count",
     "description": "Count the words in a piece of text.",
     "parameters": {"text": _p("string", description="text to count")},
     "fn": lambda args: str(len(str(args["text"]).split()))},
    {"name": "uppercase",
     "description": "Convert text to UPPERCASE.",
     "parameters": {"text": _p("string", description="text to upcase")},
     "fn": lambda args: str(args["text"]).upper()},
    {"name": "reverse",
     "description": "Reverse a piece of text.",
     "parameters": {"text": _p("string", description="text to reverse")},
     "fn": lambda args: str(args["text"])[::-1]},
    {"name": "repeat",
     "description": "Repeat a piece of text N times.",
     "parameters": {"text": _p("string", description="text to repeat"),
                    "times": _p("integer", description="how many times")},
     "fn": lambda args: str(args["text"]) * int(args["times"])},
]
```

**Why validate before running?** A tool that throws on bad input gives a cryptic traceback. Validating against the schema turns that into a precise, recoverable message like `parameter times must be integer` — which the agent can read and correct.

</details>